In [2]:
import pandas as pd

In [ ]:
df = pd.read_csv('bolsa_familia_por_estado_2023.csv')

# Verificar e multiplicar a coluna
if 'valor_medio_estadual' in df.columns:
    # Multiplicar e formatar com 2 casas decimais
    df['qtd_familias_beneficiarias_bolsa_familia_ajustado'] = (df['qtd_familias_beneficiarias_bolsa_familia_s'] / 12).round(0)
    
    # Salvar mantendo 2 casas decimais
    df.to_csv('bolsa_familia/dadosTratados/bolsa_familia_por_estado_2023.csv', index=False)
    
    print("Ajuste realizado com sucesso! Resultado com 2 casas decimais:")
    print(df[['uf_sigla', 'qtd_familias_beneficiarias_bolsa_familia_s', 'qtd_familias_beneficiarias_bolsa_familia_ajustado']].head())
else:
    print("Erro: Coluna 'valor_medio_bf' não encontrada.")
    print("Colunas disponíveis:", list(df.columns))

Ajuste realizado com sucesso! Resultado com 2 casas decimais:
  uf_sigla  qtd_familias_beneficiarias_bolsa_familia_s  \
0       AC                                     1310025   
1       AL                                     5449440   
2       AM                                     6362348   
3       AP                                     1227549   
4       BA                                    25393633   

   qtd_familias_beneficiarias_bolsa_familia_ajustado  
0                                           109169.0  
1                                           454120.0  
2                                           530196.0  
3                                           102296.0  
4                                          2116136.0  


In [47]:
# Dicionário de mapeamento de código UF para sigla e nome completo
uf_map = {
    11: {'sigla': 'RO', 'nome': 'Rondônia'},
    12: {'sigla': 'AC', 'nome': 'Acre'},
    13: {'sigla': 'AM', 'nome': 'Amazonas'},
    14: {'sigla': 'RR', 'nome': 'Roraima'},
    15: {'sigla': 'PA', 'nome': 'Pará'},
    16: {'sigla': 'AP', 'nome': 'Amapá'},
    17: {'sigla': 'TO', 'nome': 'Tocantins'},
    21: {'sigla': 'MA', 'nome': 'Maranhão'},
    22: {'sigla': 'PI', 'nome': 'Piauí'},
    23: {'sigla': 'CE', 'nome': 'Ceará'},
    24: {'sigla': 'RN', 'nome': 'Rio Grande do Norte'},
    25: {'sigla': 'PB', 'nome': 'Paraíba'},
    26: {'sigla': 'PE', 'nome': 'Pernambuco'},
    27: {'sigla': 'AL', 'nome': 'Alagoas'},
    28: {'sigla': 'SE', 'nome': 'Sergipe'},
    29: {'sigla': 'BA', 'nome': 'Bahia'},
    31: {'sigla': 'MG', 'nome': 'Minas Gerais'},
    32: {'sigla': 'ES', 'nome': 'Espírito Santo'},
    33: {'sigla': 'RJ', 'nome': 'Rio de Janeiro'},
    35: {'sigla': 'SP', 'nome': 'São Paulo'},
    41: {'sigla': 'PR', 'nome': 'Paraná'},
    42: {'sigla': 'SC', 'nome': 'Santa Catarina'},
    43: {'sigla': 'RS', 'nome': 'Rio Grande do Sul'},
    50: {'sigla': 'MS', 'nome': 'Mato Grosso do Sul'},
    51: {'sigla': 'MT', 'nome': 'Mato Grosso'},
    52: {'sigla': 'GO', 'nome': 'Goiás'},
    53: {'sigla': 'DF', 'nome': 'Distrito Federal'}
}

# Supondo que você está processando o arquivo de 2023
ano = 2023  # Altere para o ano correto conforme o arquivo

# Carregue seus dados (substitua pelo caminho real)
df = pd.read_csv("bolsa_familia/dadosLimpos/valorRepassado_familia_2023.csv")  # Ou Excel, JSON, etc.

# Extrair código UF (2 primeiros dígitos do código IBGE)
df['cod_uf'] = df['codigo_ibge'].astype(str).str[:2].astype(int)

# Adicionar sigla e nome do estado
df['uf_sigla'] = df['cod_uf'].map(lambda x: uf_map[x]['sigla'])
df['uf_nome'] = df['cod_uf'].map(lambda x: uf_map[x]['nome'])

# Agregar por estado (como o ano é único, não precisamos groupby por ano)
# Agregar por estado (contando municípios únicos)
df_estado = df.groupby(['uf_sigla', 'uf_nome']).agg({
    'codigo_ibge': 'nunique',  # Conta municípios únicos por estado
    'qtd_familias_beneficiarias_bolsa_familia_s': 'sum',
    'pbf_vlr_medio_benef_f' : 'sum'
}).reset_index()

# Renomear a coluna de contagem para ficar claro
df_estado = df_estado.rename(columns={'codigo_ibge': 'qtd_municipios'})

df_estado['valor_medio_estadual']= df_estado['pbf_vlr_medio_benef_f'] / df_estado['qtd_municipios']
df_estado['ano']= ano; 


# Calcular valor médio e adicionar coluna de ano
df_estado['ano'] = ano  # Adiciona o ano correspondente ao arquivo

# Ordenar por estado
df_estado = df_estado.sort_values('uf_sigla')

# Selecionar colunas de interesse
resultado = df_estado[['ano', 'uf_sigla', 'uf_nome', 'qtd_familias_beneficiarias_bolsa_familia_s', 'pbf_vlr_medio_benef_f','valor_medio_estadual']]

# Salvar em CSV
nome_arquivo_saida = f'bolsa_familia_por_estado_{ano}.csv'
resultado.to_csv(nome_arquivo_saida, index=False, float_format='%.2f')

print(f"Arquivo salvo: {nome_arquivo_saida}")
print(resultado.head())

Arquivo salvo: bolsa_familia_por_estado_2023.csv
    ano uf_sigla   uf_nome  qtd_familias_beneficiarias_bolsa_familia_s  \
0  2023       AC      Acre                                     1310025   
1  2023       AL   Alagoas                                     5449440   
2  2023       AM  Amazonas                                     6362348   
3  2023       AP     Amapá                                     1227549   
4  2023       BA     Bahia                                    25393633   

   pbf_vlr_medio_benef_f  valor_medio_estadual  
0              166009.08           7545.867273  
1              705893.44           6920.523922  
2              468606.41           7558.167903  
3              117496.35           7343.521875  
4             2798837.68           6711.840959  
